# Jupyter Notebook Code Compilation for Homework Reflection: Weeks 5-8

# Week 5 Homework Reflection

Import Models Used for Week 5 Homework Reflection

Question 1: Draw a diagram for the following negative feedback loop:

Sweating causes body temperature to decrease.  High body temperature causes sweating.
A negative feedback loop means that one thing increases another while the second thing decreases the first.
Remember that we are using directed acyclic graphs where two things cannot directly cause each other.

Question 2: Describe an example of a positive feedback loop.  This means that one things increases another while the second things also increases the first.

Quesion 3: Draw a diagram for the following situation:

Lightning storms frighten away deer and bears, decreasing their population, and cause flowers to grow, increasing their population. 
Bears eat deer, decreasing their population.
Deer eat flowers, decreasing their population.
Write a dataset that simulates this situation.  (Show the code.) Include noise / randomness in all cases. 
Identify a backdoor path with one or more confounders for the relationship between deer and flowers.

Question 4: 
Draw a diagram for a situation of your own invention.  The diagram should include at least four nodes, one confounder, and one collider.  Be sure that it is acyclic (no loops).  Which node would say is most like a treatment (X)?  Which is most like an outcome (Y)?

# Week 6 Homework Reflection

Import Models Used for Week 6 Homework Reflection

Question 1: What is a potential problem with computing the Marginal Treatment Effect simply by comparing each untreated item to its counterfactual and taking the maximum difference?  (Hint: think of statistics here.  Consider that only the most extreme item ends up being used to estimate the MTE.  That's not necessarily a bad thing; the MTE is supposed to come from the untreated item that will produce the maximum effect.  But there is nevertheless a problem.)
Possible answer: We are likely to find the item with the most extreme difference, which may be high simply due to randomness.
(Please explain / justify this answer, or give a different one if you can think of one.)

Question 2: Propose a solution that remedies this problem and write some code that implements your solution.  It's very important here that you clearly explain what your solution will do.
Possible answer: maybe we could take the 90th percentile of the treatment effect and use it as a proxy for the Marginal Treatment Effect.
(Either code this answer or choose a different one.)

# Week 7 Homework Reflection

Import Models Used for Week 7 Homework Reflection

Question 1: Create a linear regression model involving a confounder that is left out of the model.  Show whether the true correlation between $$X$$ and $$Y$$ is overestimated, underestimated, or neither.  Explain in words why this is the case for the given coefficients you have chosen.

Question 2:  Perform a linear regression analysis in which one of the coefficients is zero, e.g.

W = [noise]
X = [noise]
Y = 2 * X + [noise]

And compute the p-value of a coefficient - in this case, the coefficient of W.  
(This is the likelihood that the estimated coefficient would be as high or low as it is, given that the actual coefficient is zero.)
If the p-value is less than 0.05, this ordinarily means that we judge the coefficient to be nonzero (incorrectly, in this case.)
Run the analysis 1000 times and report the best (smallest) p-value.  
If the p-value is less than 0.05, does this mean the coefficient actually is nonzero?  What is the problem with repeating the analysis?

# Week 8 Homework Reflection

Import Models Used for Week 8 Homework Reflection

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.spatial.distance import mahalanobis

Question 1: Include the code you used to solve Week 8's coding quiz problems and write about the obstacles / challenges / insights you encountered while solving them.

In [ ]:
# Question 1 Code for Week 8 Coding Quiz

# Load the dataset
df = pd.read_csv('homework_8.1.csv')

# Estimating propensity scores using logistic regression
X_features = sm.add_constant(df['Z'])
logit_model = sm.Logit(df['X'], X_features).fit()
df['propensity_score'] = logit_model.predict(X_features)

# Calculating the inverse probability weights
df['ipw'] = np.where(df['X'] == 1, 
                     1 / df['propensity_score'], 
                     1 / (1 - df['propensity_score']))

# Estimating the ATE
mu_1 = np.mean(df['Y'] * df['X'] * df['ipw'])
mu_0 = np.mean(df['Y'] * (1 - df['X']) * df['ipw'])
ate = mu_1 - mu_0

# Printing Results
print(f"Calculated ATE: {ate:.2f}")

In [ ]:
# Question 2 Code for Week 8 Coding Quiz

# Load the dataset
df = pd.read_csv('homework_8.1.csv')

# Fit the logistic regression model (Z predicting X) with an intercept
X_features = sm.add_constant(df['Z'])
logit_model = sm.Logit(df['X'], X_features).fit()

# Predict the propensity scores
df['propensity_score'] = logit_model.predict(X_features)

# Display the first three values
print("First three propensity scores:")
print(df['propensity_score'].head(3).round(2).tolist())

In [ ]:
# Question 3 Code for Week 8 Coding Quiz

# Load the data
df = pd.read_csv('homework_8.2.csv')

# Compute the 2x2 covariance matrix of Z1 and Z2, and its inverse
# As instructed, stack Z1 and Z2 into a 2xN matrix
Z_matrix = df[['Z1', 'Z2']].values.T
cov_matrix = np.cov(Z_matrix)
inv_cov = np.linalg.inv(cov_matrix)

# Separate treated (X=1) and untreated (X=0) groups
treated = df[df['X'] == 1].reset_index(drop=True)
untreated = df[df['X'] == 0].reset_index(drop=True)

# Perform nearest-neighbor matching with replacement
matched_untreated_Y = []
untreated_coords = untreated[['Z1', 'Z2']].values

for i in range(len(treated)):
    treated_coord = treated.loc[i, ['Z1', 'Z2']].values
    
    # Compute Mahalanobis distance to all untreated units
    distances = [mahalanobis(treated_coord, u, inv_cov) for u in untreated_coords]
    
    # Find the single nearest untreated item
    nearest_idx = np.argmin(distances)
    matched_untreated_Y.append(untreated.loc[nearest_idx, 'Y'])

# Compute the average treatment effect across the matched pairs
ate_matched = np.mean(treated['Y']) - np.mean(matched_untreated_Y)
print(f"Calculated ATE: {ate_matched:.2f}")

In [ ]:
# Question 4 Code for Week 8 Coding Quiz

# Load the data
df = pd.read_csv('homework_8.2.csv')

# Compute the inverse covariance matrix for Z1 and Z2
Z_matrix = df[['Z1', 'Z2']].values.T
cov_matrix = np.cov(Z_matrix)
inv_cov = np.linalg.inv(cov_matrix)

# Get the coordinates of all untreated items
untreated_coords = df[df['X'] == 0][['Z1', 'Z2']].values

# Check the options provided in the question
options = {
    'A': (0.2, -0.4),
    'B': (2.3, 1.2),
    'C': (1.5, -1.3),
    'D': (0.9, 1.4)
}

for opt, vals in options.items():
    # Calculate the Mahalanobis distance from this option to all untreated items
    distances = [mahalanobis(np.array(vals), u, inv_cov) for u in untreated_coords]
    print(f"Option {opt} {vals} -> Minimum Mahalanobis distance to untreated: {min(distances):.4f}")